In [94]:

from transformers import AutoTokenizer, AutoModel
from transformers import Wav2Vec2Processor, Wav2Vec2Model,get_linear_schedule_with_warmup
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.nn import CrossEntropyLoss
import PIL.Image as Image
import numpy as np
import pandas as pd
import os
import librosa
import cv2

In [95]:
text_tokenizer = AutoTokenizer.from_pretrained("C:/EmoBase/loaded resorce/emocontext_model2")
text_model = AutoModel.from_pretrained("C:/EmoBase/loaded resorce/emocontext_model2")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5478.32it/s]
[transformers] RobertaModel LOAD REPORT from: C:/EmoBase/loaded resorce/emocontext_model2
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [96]:
audio_processor = Wav2Vec2Processor.from_pretrained("C:/EmoBase/loaded resorce/wav2vec_model")
model_audio = Wav2Vec2Model.from_pretrained("C:/EmoBase/loaded resorce/wav2vec_model")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 3373.67it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: C:/EmoBase/loaded resorce/wav2vec_model
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 
projector.weight  | UNEXPECTED |  | 
projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [97]:
image_processor= AutoImageProcessor.from_pretrained("C:/EmoBase/loaded resorce/face_model")
model_image = AutoModel.from_pretrained("C:/EmoBase/loaded resorce/face_model")

Loading weights: 100%|██████████| 318/318 [00:00<00:00, 3123.51it/s]
[transformers] ResNetModel LOAD REPORT from: C:/EmoBase/loaded resorce/face_model
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.1.bias   | UNEXPECTED |  | 
classifier.1.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [98]:
df_train = pd.read_csv("C:/EmoBase/Data/MELD.Raw/train/train_sent_emo.csv")
df_test= pd.read_csv("C:/EmoBase/Data/MELD.Raw/test/test_sent_emo.csv")
df_dev= pd.read_csv("C:/EmoBase/Data/MELD.Raw/dev/dev_sent_emo.csv")

label_map = {
    "neutral": 0, "joy": 1, "sadness": 2,
    "anger": 3, "fear": 4, "surprise": 5, "disgust": 6
}

df_train['label'] = df_train['Emotion'].map(label_map)
df_test['label'] = df_test['Emotion'].map(label_map)
df_dev['label'] = df_dev['Emotion'].map(label_map)
df_dev['label'] = df_dev['Emotion'].map(label_map)

In [99]:
def load_audio(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000)
        if len(y) == 0:
            return np.zeros(48000, dtype=np.float32)
        return y.astype(np.float32)
    except:
        return np.zeros(48000, dtype=np.float32)

In [100]:
def extract_frame(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        middle_frame = total_frames // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)
        ret, frame = cap.read()
        cap.release()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            return Image.fromarray(frame)
        else:
            return Image.new("RGB", (224, 224), color=(128, 128, 128))
    except:
        return Image.new("RGB", (224, 224), color=(128, 128, 128))

In [101]:
def get_paths(dia_id, utt_id, audio_folder, video_folder):
    filename = f"dia{dia_id}_utt{utt_id}"
    audio_path = os.path.join(audio_folder, filename + ".wav")
    video_path = os.path.join(video_folder, filename + ".mp4")
    return audio_path, video_path

In [102]:
class FusionDataset(Dataset):
    def __init__(self, df, audio_folder, video_folder, 
                 text_tokenizer, audio_processor, image_processor):
        self.df = df.reset_index(drop=True)
        self.audio_folder = audio_folder
        self.video_folder = video_folder
        self.text_tokenizer = text_tokenizer
        self.audio_processor = audio_processor
        self.image_processor = image_processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']
        label = torch.tensor(row['label'], dtype=torch.long)

        
        text = str(row['Utterance'])
        text_inputs = self.text_tokenizer(
            text,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )


        audio_path, video_path = get_paths(
            dia_id, utt_id, 
            self.audio_folder, 
            self.video_folder
        )
        audio = load_audio(audio_path)
        audio_inputs = self.audio_processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding="max_length",
            max_length=48000,
            truncation=True
        )

        frame = extract_frame(video_path)
        image_inputs = self.image_processor(
            images=frame,
            return_tensors="pt"
        )

        return {
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "input_values": audio_inputs["input_values"].squeeze(0),
            "pixel_values": image_inputs["pixel_values"].squeeze(0),
            "labels": label
        }

In [103]:
train_dataset = FusionDataset(
    df_train,
    audio_folder="C:/MELD.Raw/Audio/train",
    video_folder="C:/MELD.Raw/train/train_splits",
    text_tokenizer=text_tokenizer,
    audio_processor=audio_processor,
    image_processor=image_processor
)

dev_dataset = FusionDataset(
    df_dev,
    audio_folder="C:/MELD.Raw/Audio/dev",
    video_folder="C:/MELD.Raw/dev/dev_splits_complete",
    text_tokenizer=text_tokenizer,
    audio_processor=audio_processor,
    image_processor=image_processor
)

test_dataset = FusionDataset(
    df_test,
    audio_folder="C:/MELD.Raw/Audio/test",
    video_folder="C:/MELD.Raw/test/output_repeated_splits_test",
    text_tokenizer=text_tokenizer,
    audio_processor=audio_processor,
    image_processor=image_processor
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,num_workers=2,pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False,num_workers=2,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2,pin_memory=True)

print(len(train_dataset), len(dev_dataset), len(test_dataset))

9989 1109 2610


In [104]:
class FusionModel(torch.nn.Module):
    def __init__(self, text_model, model_audio, model_image, num_classes=7):
        super(FusionModel, self).__init__()
        self.text_model = text_model
        self.model_audio = model_audio
        self.model_image = model_image

        self.text_fc = torch.nn.Linear(768, 256)
        self.audio_fc = torch.nn.Linear(768, 256)
        self.image_fc = torch.nn.Linear(2048, 256)

        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(256 * 3, 512),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, input_values, pixel_values):
        text_out = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_features = self.text_fc(text_out.last_hidden_state[:, 0, :])

        audio_out = self.model_audio(input_values=input_values)
        audio_features = self.audio_fc(audio_out.last_hidden_state.mean(dim=1))

        image_out = self.model_image(pixel_values=pixel_values)
        image_features = self.image_fc(image_out.pooler_output.squeeze(-1).squeeze(-1))

        combined = torch.cat([text_features, audio_features, image_features], dim=1)
        output = self.classifier(combined)
        return output

In [105]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

fusion_model = FusionModel(
    text_model=text_model,
    model_audio=model_audio,
    model_image=model_image,
    num_classes=7
)

fusion_model = fusion_model.to(device)
print(f"Using: {device}")

Using: cuda


In [106]:
for param in fusion_model.text_model.parameters():
    param.requires_grad = False

for param in fusion_model.model_audio.parameters():
    param.requires_grad = False

for param in fusion_model.model_image.parameters():
    param.requires_grad = False

In [107]:
y=df_train['label'].values
class_weight=compute_class_weight(class_weight='balanced', classes=np.array([0,1,2,3,4,5,6]), y=y)
class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)
loss_fn = CrossEntropyLoss(weight=class_weight_tensor)

In [108]:
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

In [ ]:
total_steps = len(train_loader) * 10
warmup_steps = total_steps // 10
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=1e-4)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
for epoch in range(10):
    print(f"Epoch {epoch+1}/10")
    fusion_model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        input_values = batch["input_values"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = fusion_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            input_values=input_values,
            pixel_values=pixel_values
        )
        loss = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Average Loss: {avg_loss:.4f}")

Epoch 1/10


: 

In [ ]:
torch.save(fusion_model.state_dict(), "fusion_model.pth")
print("Fusion model saved!")